# 05 — Feature Engineering (FD001)

**Phase 5 — Feature Engineering**

Builds on the RUL-labeled splits from `04_rul_generation.ipynb`:
- Rolling mean / std / min / max
- Lag features
- Trend features (slope, rate of change, % change)
- Degradation features (cumulative change, EMA, health index)
- Feature selection (drop constant + redundant columns)

In [6]:
import sys
sys.path.append('..')

import pandas as pd

from src.config.config import PROCESSED_DATA_DIR, SENSOR_COLUMNS, COLUMN_NAMES
from src.features.feature_engineering import build_feature_pipeline, select_features

VARIANT = 'FD001'
train_split = pd.read_parquet(PROCESSED_DATA_DIR / f'train_{VARIANT}_rul_split_train.parquet')
val_split = pd.read_parquet(PROCESSED_DATA_DIR / f'train_{VARIANT}_rul_split_val.parquet')
test_labeled = pd.read_parquet(PROCESSED_DATA_DIR / f'test_{VARIANT}_rul_labeled.parquet')
train_split.shape, val_split.shape, test_labeled.shape

((16561, 28), (4070, 28), (13096, 27))

## Drop constant sensors first
Per the Phase 3 sensor analysis, some sensors are constant for FD001 and contribute no signal — drop them before feature engineering so we don't waste time building rolling/lag features on noise.

In [7]:
sensor_std = train_split[SENSOR_COLUMNS].std()
active_sensors = sensor_std[sensor_std > 1e-3].index.tolist()
print(f'{len(active_sensors)} / {len(SENSOR_COLUMNS)} sensors are non-constant for {VARIANT}')
print(active_sensors)

15 / 21 sensors are non-constant for FD001
['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


## Run the feature-engineering pipeline
Applied identically to train, validation, AND test — using each engine's own history only (no cross-engine leakage).

In [8]:
train_fe = build_feature_pipeline(train_split, active_sensors)
val_fe = build_feature_pipeline(val_split, active_sensors)
test_fe = build_feature_pipeline(test_labeled, active_sensors)

print('Train features shape:', train_fe.shape)
print('Val features shape  :', val_fe.shape)
print('Test features shape :', test_fe.shape)
print('NaNs in train:', train_fe.isna().sum().sum())

[2026-10-09 06:45:25] INFO     | src.features.feature_engineering | Added rolling features for 15 columns x 3 windows (180 new columns)
[2026-10-09 06:45:25] INFO     | src.features.feature_engineering | Added lag features for 15 columns x 3 lags (45 new columns)
[2026-10-09 06:45:27] INFO     | src.features.feature_engineering | Added trend features for 15 columns
[2026-10-09 06:45:27] INFO     | src.features.feature_engineering | Added degradation features for 15 columns
[2026-10-09 06:45:27] INFO     | src.features.feature_engineering | Feature pipeline complete: 343 total columns
[2026-10-09 06:45:27] INFO     | src.features.feature_engineering | Added rolling features for 15 columns x 3 windows (180 new columns)
[2026-10-09 06:45:27] INFO     | src.features.feature_engineering | Added lag features for 15 columns x 3 lags (45 new columns)
[2026-10-09 06:45:27] INFO     | src.features.feature_engineering | Added trend features for 15 columns
[2026-10-09 06:45:27] INFO     | src.feat

## Feature selection
Drop near-constant and highly-correlated (redundant) engineered features, fit ONLY on the training split (never on validation or test, to avoid leaking their distribution into the selection).

In [9]:
non_feature_cols = COLUMN_NAMES + ['RUL', 'RUL_clipped']
candidate_features = [c for c in train_fe.columns if c not in non_feature_cols]

selected_features = select_features(train_fe, candidate_features)
print(f'{len(candidate_features)} candidates -> {len(selected_features)} selected')

[2026-10-09 06:45:31] INFO     | src.features.feature_engineering | Feature selection: 315 candidates -> 232 selected (15 constant/near-constant dropped, 68 redundant dropped)
315 candidates -> 232 selected


## Persist feature-engineered datasets
Save the full engineered frames (so `RUL`/`RUL_clipped` and IDs are preserved) plus the selected feature list, so Phase 6/7 notebooks don't need to redo this work.

In [10]:
import json

train_fe.to_parquet(PROCESSED_DATA_DIR / f'train_{VARIANT}_features.parquet', index=False)
val_fe.to_parquet(PROCESSED_DATA_DIR / f'val_{VARIANT}_features.parquet', index=False)
test_fe.to_parquet(PROCESSED_DATA_DIR / f'test_{VARIANT}_features.parquet', index=False)

with open(PROCESSED_DATA_DIR / f'selected_features_{VARIANT}.json', 'w') as f:
    json.dump({'selected_features': selected_features, 'active_sensors': active_sensors}, f, indent=2)

print('Saved feature-engineered train/val/test + selected feature list to data/processed/')

Saved feature-engineered train/val/test + selected feature list to data/processed/


## Next
Continue to **06_baseline_models.ipynb** to train Linear Regression, Random Forest, XGBoost, and LightGBM on these engineered features.